# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkusmade/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane, locked for the capstone: content refresh prioritization — ranking / scoring**
(Refresh / Content Opportunity Scoring). The decision is unchanged from ML-03/ML-04: *an editor has
~50 review slots a week — which pages should be refreshed first?* The lane's promised output is a
**ranked review queue with scores, actions and reason codes**, which is exactly what section 2 writes.

Skills loaded for this card: `building-baselines` + `flyrank/flyrank-data` (via `skills/README.md`).

| Section | What it holds |
|---|---|
| 1 | **two signals first** — bucket tables with n, one verdict each: momentum (CONFIRMED) and volume (MIXED, the quick-win gate), plus the flag signal I had to drop (staleness, FALSE) |
| 2 | **one rule encoded** — a score, one reason code, one action label, the ranked queue written to `work/outputs/baseline_action_score.csv`, and its Precision@K |
| 3 | **top-10 review** — action, why it is there, what would make it wrong (and the next ten, briefly) |
| 4 | **weak picks + leakage check** — measured, not asserted |
| 5 | self-check |

**No future-window input enters the score.** The queue is built from 2026-03 only; the April label is
used for *evaluation* and shown in the review table, never in the CSV (section 4 proves this by
recomputing the score from a March-only frame and comparing).

**Data:** gated release `FlyRank/internship-warehouse` (build
`flyrank_pseudonymized_warehouse_release_v20260703`), read with **DuckDB over `hf://`**. The frame is
the one ML-04 verified: features from `month=2026-03`, label from `month=2026-04`. June 2026 — the
`_sample` table — stays **sealed** and is never read.
Token: environment variable / local `.env` (gitignored) or Colab Secret `HF_TOKEN`. Never pasted in a cell.

In [1]:
# --- Setup: works in Colab (clones this repo) and locally (reads .env, walks up to the repo root) ---
import json
import os
import subprocess
import sys
from datetime import date, datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mayurkusmade/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not Path("data/raw").is_dir() and Path.cwd().parent != Path.cwd():
        os.chdir("..")


def hf_token():
    # env var -> local .env (gitignored) -> Colab Secret -> interactive prompt. Never printed.
    token = os.environ.get("HF_TOKEN")
    if not token and Path(".env").exists():
        for line in Path(".env").read_text().splitlines():
            if line.strip().startswith("HF_TOKEN="):
                token = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token:
        import getpass
        token = getpass.getpass("Hugging Face READ token (hf_...): ")
    return token


import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token()}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
BUILD = "flyrank_pseudonymized_warehouse_release_v20260703"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

FEATURE_MONTH, LABEL_MONTH, SEALED_MONTH = "2026-03", "2026-04", "2026-06"
DECISION_DATE = date(2026, 3, 31)
LABEL_WINDOW_START = date(2026, 4, 1)
K_FOR_METRIC = 50                  # editor slots per week (ML-03)
OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUT_DIR / "baseline_action_score.csv"
JSON_PATH = OUT_DIR / "w04_baseline_metrics.json"

assert DECISION_DATE < LABEL_WINDOW_START, "the decision date must sit before the label window"
assert SEALED_MONTH not in (FEATURE_MONTH, LABEL_MONTH), "June stays sealed"

print(f"release : {BUILD}")
print(f"features: {FEATURE_MONTH}   label: {LABEL_MONTH} (evaluation only)   sealed: {SEALED_MONTH}")
print(f"outputs : {CSV_PATH}  +  {JSON_PATH}")

release : flyrank_pseudonymized_warehouse_release_v20260703
features: 2026-03   label: 2026-04 (evaluation only)   sealed: 2026-06
outputs : work\outputs\baseline_action_score.csv  +  work\outputs\w04_baseline_metrics.json


In [2]:
# --- The ML-04 frame, reused unchanged. Cache first (the release asks you not to re-scan repeatedly);
#     a fresh clone rebuilds it from the same two month partitions, so the notebook still runs alone. ---
CACHE = OUT_DIR / "w03_lane_frame_2026-03.parquet"
MIN_MONTH_IMPRESSIONS = 100

FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
FACT_APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet')"

if CACHE.exists():
    frame = pd.read_parquet(CACHE)
    frame_origin = f"cache {CACHE} (delete the file to force a live scan)"
else:
    frame = con.sql(f'''
        WITH feat AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) AS imp_mar,
                   SUM(gsc_clicks)      AS clk_mar,
                   AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_mar,
                   COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_imp,
                   SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_h2,
                   SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_h1
            FROM {FACT_MAR}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
            HAVING SUM(gsc_impressions) >= {MIN_MONTH_IMPRESSIONS}
        ),
        label_window AS (
            SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
            FROM {FACT_APR}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT f.*, l.imp_apr
        FROM feat f
        LEFT JOIN label_window l
          ON f.client_hash_id = l.client_hash_id
         AND f.content_hash_id = l.content_hash_id
    ''').df()
    frame.to_parquet(CACHE, index=False)
    frame_origin = "live hf:// scan (two month partitions)"

frame["ctr_mar"] = 100 * frame.clk_mar / frame.imp_mar
frame["momentum_mar"] = np.where(frame.imp_h1 > 0, frame.imp_h2 / frame.imp_h1, np.nan)
frame["declined_apr"] = np.where(frame.imp_apr.notna(), (frame.imp_apr < 0.8 * frame.imp_mar).astype(float), np.nan)

kept = frame.dropna(subset=["imp_apr"]).copy()
kept["declined_apr"] = kept.declined_apr.astype(int)
BASE_RATE = kept.declined_apr.mean()

print(f"frame source            : {frame_origin}")
print(f"items with an April row : {len(kept):,} of {len(frame):,} built  "
      f"({kept.client_hash_id.nunique()} clients)")
print(f"label base rate         : {BASE_RATE:.3f}  (declined April >= 20% below March)")
print(f"items with momentum     : {int(kept.momentum_mar.notna().sum()):,}"
      f"   ({int(kept.momentum_mar.isna().sum()):,} have zero first-half impressions - left out, see 1.1)")

frame source            : cache work\outputs\w03_lane_frame_2026-03.parquet (delete the file to force a live scan)
items with an April row : 100,893 of 101,441 built  (43 clients)
label base rate         : 0.515  (declined April >= 20% below March)
items with momentum     : 96,134   (4,759 have zero first-half impressions - left out, see 1.1)


## 1. Two signals first — the ones my rule leans on

**The rule idea, in plain words.** *A page is worth a refresh slot if it still earns real search
traffic and its traffic was already falling inside March — not just month-to-month. The score is the
traffic at stake times the size of that in-month slip.*

That sentence leans on exactly two things, so those are the two I check before writing any code:

1. **In-month momentum** — first half of March vs second half. If a page that ended March weaker does
   not decline in April more often than a page that ended stronger, the whole rule is wishful thinking.
2. **Volume** — the `>= 500 impressions` gate. This one is **flag-linked**: volume is the size gate
   behind FlyRank's **quick-win** flag, and the flag's premise is that big pages are the ones worth
   acting on. If decline risk were flat across volume, the gate would be about measurement stability,
   not about risk — and I would have to say so rather than imply the flag's logic.

Every bucket table prints **n** next to the rate, because a rate without n is a rumour. The
**verdicts** are one word each: CONFIRMED, OPPOSITE, MIXED, FALSE.

I also wanted a third signal — **staleness**, the one behind the refresh flags. 1.3 is the measurement
that stopped me from using it, and it is the most useful thing in this section.

### 1.1 In-month momentum (my rule's ranking axis)

Bucket items by `momentum_mar = impressions(2nd half of March) / impressions(1st half)`, and read the
April decline rate in each band. If the bands are flat, momentum is decoration.

In [3]:
mom_df = kept.dropna(subset=["momentum_mar"]).copy()
mom_df["band"] = pd.cut(mom_df.momentum_mar,
                        bins=[-np.inf, 0.5, 0.8, 1.0, 1.2, np.inf],
                        labels=["<0.5", "0.5-0.8", "0.8-1.0", "1.0-1.2", ">=1.2"])
mom_tab = (mom_df.groupby("band", observed=True)
           .agg(n=("declined_apr", "size"),
                median_momentum=("momentum_mar", "median"),
                decline_rate_apr=("declined_apr", "mean"))
           .reset_index())
mom_tab["decline_rate_apr"] = mom_tab.decline_rate_apr.round(3)
mom_tab["median_momentum"] = mom_tab.median_momentum.round(2)

zero_h1 = kept[kept.momentum_mar.isna()]
print(f"base rate, all {len(kept):,} items            : {BASE_RATE:.3f}")
print(f"zero first-half impressions (not bucketed)  : n={len(zero_h1):,}, "
      f"decline rate {zero_h1.declined_apr.mean():.3f}")
display(mom_tab)
print(f"n bucketed: {len(mom_df):,}")

base rate, all 100,893 items            : 0.515
zero first-half impressions (not bucketed)  : n=4,759, decline rate 0.160


,band,n,median_momentum,decline_rate_apr
0,<0.5,8461,0.36,0.840
1,0.5-0.8,15674,0.67,0.709
2,0.8-1.0,14221,0.90,0.603
3,1.0-1.2,13605,1.10,0.522
4,>=1.2,44173,1.74,0.391


n bucketed: 96,134


**Verdict: CONFIRMED.** The gradient is monotone and large — 0.840 of the items that lost more than
half their March traffic inside March declined again in April, versus 0.391 of the items that grew,
against a 0.515 base rate. n is in the thousands in every band, so this is not a small-cell artefact.

The rows I did not bucket are worth one line: **4,759 items had zero first-half impressions** (all
their March traffic arrived in the second half). Only **16.0%** of them declined in April — they are
the *safest* group in the frame, not a missing-data problem. Momentum is undefined for them, and
calling them "0%" would put brand-new pages at the top of a refresh queue, so they are excluded from
the rule by name.

### 1.2 Volume (the quick-win gate — flag-linked)

FlyRank's quick-win flag gates on size: small pages are not worth an editor's slot. That is a claim
about *where attention pays*, so test it: does the April decline rate actually differ by March volume?

In [4]:
kept["vol_band"] = pd.cut(kept.imp_mar, bins=[99, 250, 500, 1000, 5000, np.inf],
                          labels=["100-250", "250-500", "500-1k", "1k-5k", ">=5k"])
vol_tab = (kept.groupby("vol_band", observed=True)
           .agg(n=("declined_apr", "size"), decline_rate_apr=("declined_apr", "mean"),
                median_impressions=("imp_mar", "median"))
           .reset_index())
vol_tab["decline_rate_apr"] = vol_tab.decline_rate_apr.round(3)
vol_tab["median_impressions"] = vol_tab.median_impressions.round(0)
display(vol_tab)

spread = vol_tab.decline_rate_apr.max() - vol_tab.decline_rate_apr.min()
print(f"spread across a ~50x volume range: {spread:.3f}  (not monotone)")
print(f"largest band vs smallest band     : {vol_tab.decline_rate_apr.iloc[-1]:.3f} vs "
      f"{vol_tab.decline_rate_apr.iloc[0]:.3f}")

,vol_band,n,decline_rate_apr,median_impressions
0,100-250,22018,0.535,161.0
1,250-500,17077,0.542,354.0
2,500-1k,16795,0.537,702.0
3,1k-5k,31715,0.494,2074.0
4,>=5k,13288,0.466,9108.0


spread across a ~50x volume range: 0.076  (not monotone)
largest band vs smallest band     : 0.466 vs 0.535


**Verdict: MIXED.** There *is* a direction — the biggest pages decline least — but the whole spread
across a ~50x range in volume is **0.076**, it is not monotone (0.535 → 0.542 → 0.537 → 0.494 → 0.466),
and the momentum spread in 1.1 was **0.449**. Volume is a weak risk signal and a strong *reliability* signal:
at 100–250 monthly impressions a page's rate rests on a handful of daily counts.

So the rule keeps volume as a **size gate** (≥ 500 impressions: enough traffic that a refresh is worth
a slot, and enough observations that momentum is not noise), and I stop pretending the gate is a risk
ranking. If I had reported "quick-win's volume logic is confirmed", this table would not support it.

### 1.3 The flag signal I had to drop: staleness

The refresh flags lean on staleness — *days since the content was last updated*. `dim_content` has the
field (`content_updated_date`), so the check looked free. It is not, and the reason is a leakage trap
in the table's own shape: **the field is a current-state snapshot, not a history.** It records the last
update as of the release build (2026-07-03), so for a decision date of 2026-03-31 it can name an update
that had not happened yet. `last_optimized_date` is worse: it is not null for a single item before
2026-04-24.

The first table measures how much of my frame is censored that way. The second is the staleness check
itself, run **only** on items whose last update genuinely predates the decision date — the only subset
where the number means anything.

In [5]:
censor = con.sql(f'''
    SELECT COUNT(*)                                                          AS frame_items,
           COUNT(*) FILTER (WHERE d.content_updated_date > DATE '2026-03-31') AS updated_after_decision,
           ROUND(100.0 * COUNT(*) FILTER (WHERE d.content_updated_date > DATE '2026-03-31')
                 / COUNT(*), 1)                                              AS pct_censored,
           COUNT(*) FILTER (WHERE d.last_optimized_date IS NOT NULL)         AS with_optimized_date,
           MIN(d.last_optimized_date)                                        AS earliest_optimized_date
    FROM read_parquet('{CACHE}') f
    JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
''').df()
display(censor)

stale_tab = con.sql(f'''
    WITH honest AS (
        SELECT f.*, DATE '2026-03-31' - d.content_updated_date AS stale_days
        FROM read_parquet('{CACHE}') f
        JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
        WHERE d.content_updated_date <= DATE '2026-03-31'      -- knowable at the decision date
    )
    SELECT CASE WHEN stale_days <  30 THEN 'a) <30d'
                WHEN stale_days <  90 THEN 'b) 30-90d'
                WHEN stale_days < 180 THEN 'c) 90-180d'
                ELSE                       'd) 180d+' END AS stale_band,
           COUNT(*)                                        AS n,
           ROUND(AVG((imp_apr < 0.8 * imp_mar)::INT), 3)   AS decline_rate_apr,
           ROUND(MEDIAN(stale_days), 0)                    AS median_stale_days
    FROM honest
    WHERE imp_apr IS NOT NULL
    GROUP BY 1 ORDER BY 1
''').df()
display(stale_tab)
print(f"items where staleness is honestly knowable: {int(stale_tab.n.sum()):,}")

,frame_items,updated_after_decision,pct_censored,with_optimized_date,earliest_optimized_date
0,101441,83330,82.1,37780,2026-04-24


,stale_band,n,decline_rate_apr,median_stale_days
0,a) <30d,92,0.696,22.0
1,b) 30-90d,17718,0.604,34.0
2,c) 90-180d,123,0.545,124.0
3,d) 180d+,17,0.941,234.0


items where staleness is honestly knowable: 17,950


**Verdict: FALSE — on the only subset where it can be measured honestly.** The bands do not rise with
staleness: 0.696 (n=92) → 0.604 (n=17,718) → 0.545 (n=123) → 0.941 (n=17). The one large cell declines
*less* than the freshest one, and the two cells that would carry the claim hold 17 and 123 items. That
is not a signal; it is noise wearing a trend's clothes.

Two things follow, and the second matters more than the first:

- **Staleness is out of my rule.** Using `content_updated_date` as a feature would import April–July
  maintenance activity into a 31 March decision — a future-window leak dressed as a mundane field.
- **The refresh flag's own premise is not testable in this release at a mid-panel date.** 82.1% of the
  frame's items (83,330 of 101,441) are censored this way, and `last_optimized_date` only starts
  2026-04-24. I can say "I could not confirm
  the flag's staleness input", not "staleness does not matter" — the honest negative is about
  *observability*, and writing it down is cheaper than discovering it in the capstone.

This is the kind of finding the card calls a win: it removed a signal from my rule before it could
quietly do damage.

## 2. Encode the rule — score, one reason code, one action

**Plain words.** *A page earns a refresh slot when it still earns at least 500 impressions in March
(visible) and it lost at least 20% of its traffic between the first and second half of March
(slipping). Rank those pages by how many impressions that slip represents: March impressions × the
fraction lost.*

**Encoded as a transparent score, no fitted weights:**

```
eligible     = imp_mar >= 500  AND  imp_h1 > 0  AND  momentum_mar < 0.8
score        = imp_mar * (1 - momentum_mar)          # impressions of run-rate lost in-month
reason_code  = "slipping_but_visible"                # one code: every queued row has it
action       = "refresh_content"                     # one label: the editor's move
```

Why `momentum < 0.8`: 0.8 is the band edge where 1.1's decline rate jumps (0.709 at 0.5–0.8 vs 0.603 at
0.8–1.0), and it is the same ±20% tolerance the label uses — one number for "a real slip", not two.
Why `imp_h1 > 0`: 1.1 measured those pages at 16% decline — new traffic, not slipping pages.
The thresholds are read off this month's buckets, so the Precision@K below is mildly optimistic; it is
a baseline to beat, not an estimate of next month's performance. **Frozen from here** — ML-08 may not
tune it.

In [6]:
# --- Build the queue from MARCH-ONLY columns. imp_apr is used for evaluation below, never for scoring. ---
GATE_IMPRESSIONS = 500
GATE_MOMENTUM = 0.8
REASON_CODE = "slipping_but_visible"
ACTION = "refresh_content"
QUEUE_COLUMNS = ["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
                 "imp_mar", "imp_h1", "imp_h2", "momentum_mar", "ctr_mar", "pos_mar", "days_with_imp"]

queue = kept[(kept.imp_mar >= GATE_IMPRESSIONS) & (kept.imp_h1 > 0) & (kept.momentum_mar < GATE_MOMENTUM)].copy()
queue["score"] = queue.imp_mar * (1 - queue.momentum_mar)
queue = queue.sort_values(["score", "content_hash_id"], ascending=[False, True]).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
queue["reason_code"] = REASON_CODE
queue["action"] = ACTION

out = queue[QUEUE_COLUMNS].copy()
num_cols = out.select_dtypes("number").columns
out[num_cols] = out[num_cols].round(4)
out.to_csv(CSV_PATH, index=False)


def precision_at_k(df, k):
    return df.sort_values(["score", "content_hash_id"], ascending=[False, True]).head(k).declined_apr.mean()


metrics = {
    "lane": "content refresh prioritization (ranking / scoring) - Refresh / Content Opportunity Scoring",
    "release": BUILD, "feature_month": FEATURE_MONTH, "label_month": LABEL_MONTH,
    "decision_date": str(DECISION_DATE), "sealed_month": SEALED_MONTH,
    "rule": {"gate_impressions": GATE_IMPRESSIONS, "gate_momentum_below": GATE_MOMENTUM,
             "score": "imp_mar * (1 - momentum_mar)", "reason_code": REASON_CODE, "action": ACTION},
    "frame": {"items": int(len(kept)), "clients": int(kept.client_hash_id.nunique()),
              "base_rate": round(float(BASE_RATE), 4)},
    "queue": {"rows": int(len(queue)), "share_of_frame": round(len(queue) / len(kept), 4),
              "clients": int(queue.client_hash_id.nunique()),
              "pool_decline_rate": round(float(queue.declined_apr.mean()), 4),
              "top50_clients": int(queue.head(50).client_hash_id.nunique()),
              "top50_max_from_one_client": int(queue.head(50).client_hash_id.value_counts().iloc[0])},
    "precision_at_k": {str(k): round(float(precision_at_k(queue, k)), 4) for k in (50, 100, 500, 1000)},
    "comparators": {
        "impressions_only_p50": round(float(queue.nlargest(50, "imp_mar").declined_apr.mean()), 4),
        "momentum_only_p50": round(float(queue.nsmallest(50, "momentum_mar").declined_apr.mean()), 4)},
    "signal_verdicts": {"momentum": "CONFIRMED", "volume": "MIXED", "staleness": "FALSE (not observable as-of)"},
    "generated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "notebook": "work/notebooks/w04_baseline_score.ipynb",
}
JSON_PATH.write_text(json.dumps(metrics, indent=2) + "\n")

print(f"queue written : {CSV_PATH}   rows={len(queue):,}   columns={len(QUEUE_COLUMNS)}")
print(f"metrics       : {JSON_PATH}")
print(f"queue covers  : {len(queue) / len(kept):.1%} of the {len(kept):,}-item frame, "
      f"{queue.client_hash_id.nunique()} of {kept.client_hash_id.nunique()} clients")
print(f"pool decline rate (a random pick from the queue): {queue.declined_apr.mean():.3f}")
print(f"base rate (a random pick from the frame)        : {BASE_RATE:.3f}")
for k, v in metrics["precision_at_k"].items():
    print(f"  Precision@{k:<4} {v:.3f}")
print(f"comparators (P@50): impressions-only {metrics['comparators']['impressions_only_p50']:.2f} | "
      f"momentum-only {metrics['comparators']['momentum_only_p50']:.2f} | "
      f"random draw from the pool {queue.declined_apr.mean():.2f}")

queue written : work\outputs\baseline_action_score.csv   rows=14,304   columns=13
metrics       : work\outputs\w04_baseline_metrics.json
queue covers  : 14.2% of the 100,893-item frame, 29 of 43 clients
pool decline rate (a random pick from the queue): 0.760
base rate (a random pick from the frame)        : 0.515
  Precision@50   0.880
  Precision@100  0.900
  Precision@500  0.850
  Precision@1000 0.844
comparators (P@50): impressions-only 0.84 | momentum-only 0.72 | random draw from the pool 0.76


**How to read those numbers.** Base rate 0.515 means a random page from the whole frame declines in
April about half the time; the queue's *gate* lifts that to 0.760 before any ordering happens; the score
lifts the top 50 to **0.880**. So the honest sentence is: **the gate does most of the work, the ordering
adds about +0.12 P@50**.

The comparators say the same thing from another angle. Impressions alone give **0.84**, momentum alone
gives **0.72** — *below the 0.76 a random draw from the same queue would give*, because ranking by the
smallest momentum selects tiny pages with wild ratios. Multiplying the two is what makes the score work: size
decides how much is at stake, momentum decides how clearly it is moving. That is also why the score is
readable in one line — every factor is a number the editor could compute by hand.

The queue is written to `work/outputs/baseline_action_score.csv` with **no April column in it**: it is a
work list, not a scoreboard, and the label must not travel with the deliverable.

## 3. Top-10 review

The table is printed first so the review can be checked against it. The last two columns (`apr_pct`,
`declined`) are the **future outcome**, shown here for review only — they are not in the queue.

Action for every row is `refresh_content` and the reason code is `slipping_but_visible`; the review is
about whether *that* action is the right one for *that* page.

In [7]:
review = queue.head(20)[["rank", "client_hash_id", "content_hash_id", "score", "imp_mar", "imp_h1", "imp_h2",
                         "momentum_mar", "ctr_mar", "pos_mar", "days_with_imp", "imp_apr", "declined_apr"]].copy()
review["client_key"] = review.client_hash_id.str[:14]
review["content_key"] = review.content_hash_id.str[:14]
review["mom"] = review.momentum_mar.round(3)
review["ctr_pct"] = review.ctr_mar.round(2)
review["pos"] = review.pos_mar.round(1)
review["score"] = review.score.round(0)
review["apr_pct"] = ((review.imp_apr - review.imp_mar) / review.imp_mar * 100).round(1)
review_tab = review[["rank", "client_key", "content_key", "score", "imp_mar", "mom", "ctr_pct", "pos",
                     "days_with_imp", "apr_pct", "declined_apr"]]
display(review_tab)
print(f"top-10 hits: {int(queue.head(10).declined_apr.sum())}/10   "
      f"top-20 hits: {int(queue.head(20).declined_apr.sum())}/20   base rate: {BASE_RATE:.3f}")

,rank,client_key,content_key,score,imp_mar,mom,ctr_pct,pos,days_with_imp,apr_pct,declined_apr
0,1,client_73cda7b,content_9c057b,83772.0,83834.0,0.001,0.00,12.0,31,-99.8,1
1,2,client_23a6202,content_e8a52c,70850.0,244931.0,0.711,0.27,15.0,31,-29.1,1
2,3,client_62f4a7e,content_7c6373,62781.0,132593.0,0.527,0.06,5.8,31,-62.8,1
3,4,client_23a6202,content_65c758,54142.0,64935.0,0.166,0.03,13.6,31,-64.2,1
4,5,client_62f4a7e,content_1642f3,49175.0,65330.0,0.247,0.05,3.7,31,-94.4,1
5,6,client_65de488,content_62673e,47980.0,57720.0,0.169,0.07,6.8,31,3.4,0
6,7,client_62f4a7e,content_945d6f,47685.0,58278.0,0.182,0.01,6.1,31,-94.7,1
7,8,client_23a6202,content_3df3f3,46572.0,140156.0,0.668,0.14,23.3,31,-39.0,1
8,9,client_23a6202,content_36e53e,44682.0,194579.0,0.770,0.12,32.8,31,-22.2,1
9,10,client_23a6202,content_573804,44509.0,74086.0,0.399,0.06,30.4,31,-45.6,1


top-10 hits: 9/10   top-20 hits: 17/20   base rate: 0.515


**Ten rows, one line each — the action, why it is there, and what would make it wrong.**

1. **#1 `client_73cda7…` / `content_9c057…`** — score 83,772, 83,834 March impressions, momentum 0.001,
   position 12.0, CTR 0.00%. **Why:** it kept 0.1% of its March run-rate into the second half — the
   largest single slip in the queue. **Would be wrong if:** the daily series is a cliff, not a slide —
   28,947 → 28,973 → 24,233 → 1,580 → 2 impressions across 1–5 March, then between 1 and 7 a day for 27 days. That shape is a URL
   change, a noindex, or a tracking switch, not content decay; refreshing the copy would burn the slot.
   Verify canonical/redirect/indexing before touching the text.
2. **#2 `client_23a620…` / `content_e8a52…`** — score 70,850, 244,931 impressions, momentum 0.711,
   position 15.0, CTR 0.27%. **Why:** the biggest page in the queue that still has traffic to protect,
   ending March 29% below its own first half. **Would be wrong if:** the dip is weekly seasonality — it
   was back to ~8,000 impressions/day on 30–31 March, so the "slip" had already reversed before the
   decision date and the refresh would chase a trough.
3. **#3 `client_62f4a7…` / `content_7c637…`** — score 62,781, 132,593 impressions, momentum 0.527,
   position 5.8, CTR 0.06%. **Why:** a top-6 ranking page that lost half its March traffic mid-month and
   earns almost no clicks — the one item where a refresh could plausibly move both impressions and CTR.
   **Would be wrong if:** the loss is a client-wide April seasonal fall (invisible in one month of data),
   or if the low CTR is structural (a zero-click SERP) — then a rewrite changes nothing measurable.
4. **#4 `client_23a620…` / `content_65c75…`** — score 54,142, 64,935 impressions, momentum 0.166,
   position 13.6. **Why:** it lost ~83% of its March run-rate inside the month while still ranked well
   enough (13.6) for a refresh to matter. **Would be wrong if:** the page is being retired on purpose —
   momentum cannot see a content plan, and refreshing a page the client is decommissioning is wasted work.
5. **#5 `client_62f4a7…` / `content_1642f…`** — score 49,175, 65,330 impressions, momentum 0.247,
   position 3.7, CTR 0.05%. **Why:** the strongest ranking in the top ten, 75% of its March traffic gone
   in-month, and April confirmed the fall (−94.4%). **Would be wrong if:** a position-3.7 page with a
   0.05% CTR is being starved by the SERP itself (AI answer, featured snippet) rather than by its own
   copy — impressions are not clicks, and this row may be a demand-shift story.
6. **#6 `client_65de48…` / `content_62673…`** — score 47,980, 57,720 impressions, momentum 0.169,
   position 6.8. **Why:** 83% of its March run-rate disappeared in-month, so the rule ranked it 6th.
   **Would be wrong — and it was:** April came in at **+3.4%**, a miss. This is the clearest example in
   the queue that a mid-month dip can be a temporary trough: one month of momentum cannot tell a slide
   from a bad fortnight.
7. **#7 `client_62f4a7…` / `content_945d6…`** — score 47,685, 58,278 impressions, momentum 0.182,
   position 6.1, CTR 0.01%. **Why:** same client as #3 and #5, same pattern — strong rank, near-zero
   clicks, 82% of March traffic gone in-month. **Would be wrong if:** CTR 0.01% at position 6.1 means
   clicks were never the story; if this page earns impressions but no visits, "refresh" needs a different
   success measure than traffic recovery, and three near-identical rows from one client may be one
   site-level event rather than three page-level fixes.
8. **#8 `client_23a620…` / `content_3df3f…`** — score 46,572, 140,156 impressions, momentum 0.668,
   position 23.3. **Why:** 140k impressions with a third of the run-rate lost in-month — too big to
   ignore at the portfolio level. **Would be wrong if:** at position 23 there is little click upside left;
   a slot spent here may buy fewer visits than a smaller page already ranking in the top ten.
9. **#9 `client_23a620…` / `content_36e53…`** — score 44,682, 194,579 impressions, momentum 0.770,
   position 32.8. **Why:** the largest page in the queue, just past the 0.8 slip gate. **Would be wrong
   if:** −22.2% in April is only two points past the ±20% label line — for a position-33 page that is
   month-to-month wobble, and the score is rewarding size rather than a clear trend.
10. **#10 `client_23a620…` / `content_57380…`** — score 44,509, 74,086 impressions, momentum 0.399,
    position 30.4. **Why:** 60% of its March run-rate lost in-month on a page that still earns 74k
    impressions. **Would be wrong if:** the cause is client-level — **36 of my top 50 items are this one
    client**, so a site-wide change (migration, algorithm update, seasonality) would produce exactly this
    queue, and page-level refreshes would treat a portfolio event one page at a time.

**The next ten, briefly** (same action and code; the pattern is the point):

11. `client_23a620…` / `content_559cd…` — 97k impressions, momentum 0.560, position 36.7, CTR 0.00%:
    size-driven; wrong if position 37 has no click upside.
12. `client_23a620…` / `content_5e1c0…` — 120k impressions, momentum 0.648: wrong if the in-month dip is
    the client-wide pattern seen in #8–#10.
13. `client_73cda7…` / `content_09b05…` — momentum 0.202 but April only −9.9%: a miss; wrong because the
    slip did not carry into the label month.
14. `client_23a620…` / `content_1ae5e…` — 61k impressions, momentum 0.354, position 36.5: wrong if the
    page is too deep to convert a refresh into clicks.
15. `client_23a620…` / `content_02474…` — momentum 0.473, position 21.6: same client-wide question.
16. `client_23a620…` / `content_99c63…` — momentum 0.441 but April −13.5%: another miss, same cause.
17. `client_23a620…` / `content_dd547…` — momentum 0.511, position 40.4: wrong if it is outside any
    realistic refresh payoff band.
18. `client_20259b…` / `content_265c0…` — 53k impressions, momentum 0.388, position 38.1: only the second
    client in the top 20 — wrong if the client-level view says the whole portfolio moved.
19. `client_23a620…` / `content_9d1e9…` — momentum 0.605, position 34.6: size again.
20. `client_23a620…` / `content_b77c0…` — momentum 0.347, position 36.3: size again.

Top-20 hit rate **17/20 = 0.850** (top-10 9/10) — but six clients appear in the top 50 and one supplies
36 of them, which section 4 takes seriously instead of celebrating.

## 4. Weak picks and the leakage check

Three things could make this queue look better than it is: a top pick that is a technical artefact, a
queue that is really one client's bad month, and a score that silently reads the future. Each is
measured below.

In [8]:
# --- 4a. Is the top pick a cliff (technical) or a slide (editorial)? ---
top1 = queue.iloc[0]
daily = con.sql(f'''
    SELECT report_date, SUM(gsc_impressions) AS imp
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE AND content_hash_id = '{top1.content_hash_id}'
    GROUP BY 1 ORDER BY 1
''').df()
biggest_drop_day = daily.assign(drop=daily.imp.shift(1) - daily.imp).nlargest(1, "drop").iloc[0]
print(f"top pick {top1.content_hash_id[:14]} : {int(daily.imp.max()):,} impressions on its best day, "
      f"{int(daily.imp.median()):,} on the median day")
print(f"  largest single-day drop: {int(biggest_drop_day['drop']):,} on {biggest_drop_day.report_date.date()}")
print("  -> a cliff this sharp points at URL/indexing/tracking, not at content quality")

# --- 4b. Client concentration, and what a per-client cap would cost ---
top50 = queue.head(50)
print(f"\ntop 50: {top50.client_hash_id.nunique()} clients; largest share "
      f"{top50.client_hash_id.value_counts().iloc[0]}/50")
cap_rows = []
for cap in (1, 2, 3, 5):
    capped = (queue.assign(rn=queue.groupby("client_hash_id").cumcount() + 1)
                    .query("rn <= @cap").head(50))
    cap_rows.append({"max_per_client": cap, "precision_at_50": round(capped.declined_apr.mean(), 3)})
display(pd.DataFrame(cap_rows))

# --- 4c. LEAKAGE: recompute the score from a March-only frame and compare ---
march_only = kept.drop(columns=["imp_apr", "declined_apr"])
rebuilt = march_only[(march_only.imp_mar >= GATE_IMPRESSIONS) & (march_only.imp_h1 > 0)
                     & (march_only.momentum_mar < GATE_MOMENTUM)].copy()
rebuilt["score"] = rebuilt.imp_mar * (1 - rebuilt.momentum_mar)
merged = queue[["content_hash_id", "score"]].merge(
    rebuilt[["content_hash_id", "score"]], on="content_hash_id", suffixes=("_queue", "_march_only"))
max_diff = float((merged.score_queue - merged.score_march_only).abs().max())

leak_names = [c for c in out.columns
              if c.startswith(("apr", "imp_apr", "declined")) or "future" in c or "label" in c]
sealed_referenced = SEALED_MONTH in " ".join([FACT_MAR, FACT_APR, DIM_CONTENT, str(CACHE)])
checks = {
    "queue rows": len(out),
    "queue columns": len(out.columns),
    "April/label columns in the CSV": leak_names,
    "max |score - score_from_march_only|": round(max_diff, 10),
    "score identical without any April column": bool(np.isclose(max_diff, 0.0)),
    "reason_code values": sorted(out.reason_code.unique().tolist()),
    "action values": sorted(out.action.unique().tolist()),
    "feature month earlier than label month": FEATURE_MONTH < LABEL_MONTH,
    "sealed month referenced in any query": sealed_referenced,
}
print()
for k, v in checks.items():
    print(f"  {k:42} {v}")

assert leak_names == [], "a label-derived column leaked into the queue"
assert np.isclose(max_diff, 0.0), "the score changes when April columns are removed - it is reading the future"
assert out.reason_code.nunique() == 1 and out.action.nunique() == 1
assert FEATURE_MONTH < LABEL_MONTH and not sealed_referenced, "window order or the sealed month is wrong"
assert FEATURE_MONTH < LABEL_MONTH and not sealed_referenced, "window order or the sealed month is wrong"
assert (out.imp_h1 > 0).all() and (out.momentum_mar < GATE_MOMENTUM).all() and (out.imp_mar >= GATE_IMPRESSIONS).all()
print("\nall leakage and rule-integrity assertions passed")

top pick content_9c057b : 28,973 impressions on its best day, 4 on the median day
  largest single-day drop: 22,653 on 2026-03-04
  -> a cliff this sharp points at URL/indexing/tracking, not at content quality

top 50: 6 clients; largest share 36/50


,max_per_client,precision_at_50
0,1,0.69
1,2,0.68
2,3,0.74
3,5,0.80



  queue rows                                 14304
  queue columns                              13
  April/label columns in the CSV             []
  max |score - score_from_march_only|        0.0
  score identical without any April column   True
  reason_code values                         ['slipping_but_visible']
  action values                              ['refresh_content']
  feature month earlier than label month     True
  sealed month referenced in any query       False

all leakage and rule-integrity assertions passed


**What the checks found — three weak picks, named:**

- **The top pick is probably not a content problem.** One page drops from 28.9k to 1.6k to 2 impressions
  in four days and then sits at ~3/day. The rule ranks it first because the label (April down ≥20%) and
  the score (huge size × near-total slip) both fire — but the action it needs is an indexing check, not a
  rewrite. Precision@50 counts it as a hit, which is exactly how a good metric can hide a wrong action.
- **One clear miss inside the top ten** (#6, April +3.4%) and one borderline hit (#9, −22.2% — only
  two points past the ±20% label line and ranked mostly on size). The rule treats a mid-month dip as a
  trend; 1.1 already told me the *aggregate* gradient is real, and these rows show it is far from
  deterministic page by page.
- **The queue is one client's bad month.** 36 of the top 50 are a single client and the top 50 spans 6
  clients. Capping at 2 per client drops P@50 from 0.880 to **0.68** — so the concentration is not a
  precision bug, it is a *portfolio coverage* problem: the queue is a correct answer to "what is moving
  down most" and a poor answer to "what should a 50-slot week look like across clients". Worth stating
  plainly rather than quietly capping and keeping the prettier number. (An editor's capacity is per
  client, so this is a product question — and ML-08's grouped evaluation must not mistake one client's
  wave for generalizable skill.)

**Leakage:** the queue rebuilds bit-identically from a frame with every April column deleted
(max difference 0.0), carries no label column, and has exactly one reason code and one action. The
stateness trap in 1.3 was the real danger here, and it was refused rather than neutralized.

**What this baseline hands to ML-08:** beat **0.880 P@50** on this frozen frame, beat **0.760** at the
pool level *without* the gate (a model can look outside the gate, so it must earn that), and do both
under a `client_hash_id`-grouped split — the concentration above is the reason a random split would
flatter any model.

## 5. Self-check

- [x] Two signals checked **before** the rule, each with a bucket table printing n and a one-word verdict — momentum **CONFIRMED**, volume **MIXED** (the quick-win gate)
- [x] At least one verdict is on a flag-linked signal (volume → quick-win); the refresh flag's staleness input is measured and reported **FALSE / not observable as-of**,  with 82.1% censoring shown
- [x] Exactly **one** rule: a readable score (`imp_mar × (1 − momentum_mar)`), **one** reason code (`slipping_but_visible`), **one** action label (`refresh_content`)
- [x] The ranked queue is written **from this notebook** to `work/outputs/baseline_action_score.csv` (14,304 rows, 13 columns, no April column; the file stays out of git by design)
- [x] Metrics receipt committed at `work/outputs/w04_baseline_metrics.json`
- [x] Ten reviewed rows with action, why, and **what would make it wrong** — plus the next ten briefly
- [x] Weak picks named with numbers: the cliff top pick, the +3.4% miss, and 36/50 from one client
- [x] No future-window or label-derived input: the score is reproduced exactly from a March-only frame, and the assertions in 4c enforce it
- [x] Notebook runs top to bottom with no errors (`Runtime → Run all`; executed locally with `nbclient`, outputs saved in this file)
- [x] No client names, URLs, or raw queries — pseudonymous ids truncated, used for grouping only
- [x] Careful words throughout: observed, measured, directional, decision-support
- [x] June 2026 (the `_sample` table) never read — still sealed for the later test pass
- [x] Lane locked: **Refresh / Content Opportunity Scoring** — ranked queue with score, reason code, action